<a href="https://colab.research.google.com/github/SejalB15/Customer-churn-prediction-system/blob/main/CNN_Plant_Disease_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment No: 3A
## Convolutional Neural Network (CNN) - Plant Disease Detection
**Aim:** Use any dataset of plant disease and design a plant disease detection system using CNN.

**Dataset Used:** PlantVillage Dataset (via Kaggle)

---

## Step 1: Mount Google Drive & Install Dependencies
> Run this cell if using **Google Colab**. Skip if running locally.

In [ ]:
# Run only on Google Colab
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Download the Dataset from Kaggle

### 📥 How to Get the Dataset:
1. Go to: https://www.kaggle.com/datasets/emmarex/plantdisease
2. Create a free Kaggle account (if you don't have one)
3. Click **Download** — it will give you a ZIP file (`plantdisease.zip`)
4. Extract it — you will find a folder called **`PlantVillage`**
5. Upload the `PlantVillage` folder to your **Google Drive** under `My Drive/PlantVillage/`

**OR use the Kaggle API in Colab (recommended):**

In [ ]:
# Option A: Download using Kaggle API (recommended for Colab)
# First, upload your kaggle.json API token file

from google.colab import files
files.upload()  # Upload your kaggle.json here

import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.system('cp kaggle.json /root/.kaggle/')
os.system('chmod 600 /root/.kaggle/kaggle.json')

# Download the dataset
os.system('kaggle datasets download -d emmarex/plantdisease --unzip -p /content/PlantVillage')
print("Dataset downloaded successfully!")

## Step 3: Import Required Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TensorFlow Version:", tf.__version__)
print("All libraries imported successfully!")

## Step 4: Configuration & Dataset Path

In [ ]:
# -----------------------------------------------
# CHANGE THIS PATH to where your dataset is stored
# -----------------------------------------------
DATASET_PATH = '/content/PlantVillage'  # If using Kaggle API download
# DATASET_PATH = '/content/drive/MyDrive/PlantVillage'  # If uploaded to Google Drive

# Hyperparameters
IMG_SIZE    = 128          # Resize images to 128x128
BATCH_SIZE  = 32
EPOCHS      = 15
NUM_CLASSES = None         # Will be set automatically based on dataset
RANDOM_SEED = 42

print(f"Dataset path: {DATASET_PATH}")
print(f"Image size  : {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch size  : {BATCH_SIZE}")
print(f"Epochs      : {EPOCHS}")

## Step 5: Explore the Dataset

In [ ]:
# List all disease categories
categories = sorted(os.listdir(DATASET_PATH))
NUM_CLASSES = len(categories)

print(f"Total number of disease classes: {NUM_CLASSES}")
print("\nClasses found:")
for i, cat in enumerate(categories):
    count = len(os.listdir(os.path.join(DATASET_PATH, cat)))
    print(f"  [{i:02d}] {cat:45s} -> {count} images")

In [ ]:
# Visualize sample images from the dataset
from tensorflow.keras.preprocessing import image as keras_image

fig, axes = plt.subplots(3, 5, figsize=(18, 10))
fig.suptitle('Sample Images from PlantVillage Dataset', fontsize=16, fontweight='bold')

selected = categories[:15]
for ax, cat in zip(axes.flatten(), selected):
    cat_path = os.path.join(DATASET_PATH, cat)
    img_file = os.listdir(cat_path)[0]
    img = keras_image.load_img(os.path.join(cat_path, img_file), target_size=(IMG_SIZE, IMG_SIZE))
    ax.imshow(img)
    ax.set_title(cat.replace('_', '\n'), fontsize=7)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Step 6: Data Preprocessing & Augmentation

In [ ]:
# Training data augmentation (as mentioned in theory)
train_datagen = ImageDataGenerator(
    rescale=1.0/255,           # Normalize pixel values to [0,1]
    rotation_range=30,          # Rotate image randomly (Theory: Rotation)
    brightness_range=[0.7,1.3], # Vary brightness (Theory: Brightness)
    shear_range=0.2,            # Apply shear transformation (Theory: Shear)
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest',
    validation_split=0.2        # 80% train, 20% validation
)

# Validation data: only rescaling (no augmentation)
val_datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=0.2
)

# Load training data
train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=RANDOM_SEED
)

# Load validation data
val_generator = val_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=RANDOM_SEED
)

print(f"\nTraining samples   : {train_generator.samples}")
print(f"Validation samples : {val_generator.samples}")
print(f"Number of classes  : {train_generator.num_classes}")
NUM_CLASSES = train_generator.num_classes

## Step 7: Build the CNN Model
### CNN Architecture:
```
Input (128x128x3)
  → Conv2D + BN + MaxPool  (Feature Extraction Layer 1)
  → Conv2D + BN + MaxPool  (Feature Extraction Layer 2)
  → Conv2D + BN + MaxPool  (Feature Extraction Layer 3)
  → Conv2D + BN + MaxPool  (Feature Extraction Layer 4)
  → Flatten
  → Dense(512) + Dropout
  → Dense(NUM_CLASSES, softmax)  (Classification)
```

In [ ]:
def build_cnn_model(num_classes, img_size):
    model = models.Sequential([

        # --- Block 1: Feature Extraction ---
        layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                      input_shape=(img_size, img_size, 3)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # --- Block 2: Feature Extraction ---
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # --- Block 3: Feature Extraction ---
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # --- Block 4: Feature Extraction ---
        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # --- Flatten & Fully Connected Layers ---
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')  # Output layer
    ])
    return model

model = build_cnn_model(NUM_CLASSES, IMG_SIZE)
model.summary()

## Step 8: Compile the Model

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

print("Model compiled successfully!")

## Step 9: Train the Model

In [ ]:
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print("\nTraining complete!")

## Step 10: Plot Training Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy Plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy', color='blue')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', color='orange')
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss Plot
axes[1].plot(history.history['loss'], label='Train Loss', color='red')
axes[1].plot(history.history['val_loss'], label='Val Loss', color='green')
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Training curves saved as training_curves.png")

## Step 11: Evaluate the Model

In [ ]:
# Evaluate on validation set
val_loss, val_accuracy = model.evaluate(val_generator, verbose=1)
print(f"\n{'='*40}")
print(f"Validation Loss     : {val_loss:.4f}")
print(f"Validation Accuracy : {val_accuracy*100:.2f}%")
print(f"{'='*40}")

## Step 12: Classification Report & Confusion Matrix

In [ ]:
# Get predictions
print("Generating predictions...")
val_generator.reset()
y_pred_probs = model.predict(val_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = val_generator.classes

class_labels = list(val_generator.class_indices.keys())

# Classification Report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_labels))

In [ ]:
# Confusion Matrix (showing first 15 classes for readability)
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(16, 14))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_labels,
    yticklabels=class_labels
)
plt.title('Confusion Matrix - Plant Disease Detection', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=90, fontsize=6)
plt.yticks(rotation=0, fontsize=6)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix saved as confusion_matrix.png")

## Step 13: Predict on a Single Image

In [ ]:
def predict_disease(img_path, model, class_labels, img_size=128):
    """
    Predict the disease of a plant leaf from an image path.
    Returns the predicted class name and confidence score.
    """
    img = keras_image.load_img(img_path, target_size=(img_size, img_size))
    img_array = keras_image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array, verbose=0)
    predicted_idx = np.argmax(predictions[0])
    confidence = predictions[0][predicted_idx] * 100
    predicted_class = class_labels[predicted_idx]

    # Display result
    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.title(f"Predicted: {predicted_class}\nConfidence: {confidence:.2f}%",
              fontsize=12, fontweight='bold',
              color='green' if 'healthy' in predicted_class.lower() else 'red')
    plt.axis('off')
    plt.show()

    return predicted_class, confidence


# -------------------------------------------------------
# Test with a sample image from the validation generator
# -------------------------------------------------------
val_generator.reset()
sample_batch, sample_labels = next(val_generator)

# Save one image temporarily and predict
sample_img = (sample_batch[0] * 255).astype(np.uint8)
from PIL import Image as PILImage
pil_img = PILImage.fromarray(sample_img)
pil_img.save('/tmp/test_leaf.jpg')

true_label = class_labels[np.argmax(sample_labels[0])]
print(f"True Label: {true_label}")

pred_class, confidence = predict_disease('/tmp/test_leaf.jpg', model, class_labels, IMG_SIZE)
print(f"Predicted : {pred_class} ({confidence:.2f}% confidence)")

## Step 14: Save the Trained Model

In [ ]:
# Save model in Keras format
model.save('plant_disease_cnn_model.keras')
print("Model saved as 'plant_disease_cnn_model.keras'")

# Save class labels
import json
with open('class_labels.json', 'w') as f:
    json.dump(class_labels, f, indent=2)
print("Class labels saved as 'class_labels.json'")

# Copy to Google Drive (optional)
# import shutil
# shutil.copy('plant_disease_cnn_model.keras', '/content/drive/MyDrive/')
# print("Model copied to Google Drive!")

---
## Conclusion

In this experiment, we successfully built a **Convolutional Neural Network (CNN)** for plant disease detection using the **PlantVillage dataset**. The key steps were:

1. **Dataset Exploration** – Examined 38 classes of healthy and diseased plant leaf images
2. **Data Augmentation** – Applied rotation, brightness variation, shear, zoom, and flipping to increase dataset variety
3. **CNN Architecture** – Built a 4-block CNN with Conv2D, BatchNormalization, MaxPooling, and Dropout layers
4. **Training** – Trained for up to 15 epochs with EarlyStopping and ReduceLROnPlateau callbacks
5. **Evaluation** – Measured accuracy, loss, and visualized a Confusion Matrix and Classification Report
6. **Prediction** – Demonstrated single-image prediction with confidence score

The CNN model successfully learns visual features from leaf images to classify plant diseases, proving that deep learning is an effective tool for automated plant disease detection.

---
## Questions (For Viva)

**Q1. Discuss any other two datasets with their attributes of plant leaves disease:**
- **Mendeley Rice Leaf Disease Dataset**: Contains images of rice leaves affected by Bacterial Leaf Blight, Brown Spot, and Leaf Smut
- **iPlant Dataset**: Contains images of wheat, maize, and potato diseases with segmentation masks

**Q2. Various diseases in plants:**
- Fungal: Powdery Mildew, Rust, Blight, Leaf Spot, Scab
- Bacterial: Bacterial Blight, Crown Gall, Canker
- Viral: Mosaic Virus, Leaf Curl
- Nutritional: Chlorosis (yellowing due to nutrient deficiency)

**Q3. How CNN involves in plant disease detection:**
- CNN extracts spatial features (edges, textures, spots) from leaf images automatically
- Convolutional layers detect patterns like spots, discoloration, and lesions
- Pooling layers reduce spatial dimensions while preserving important features
- Fully connected layers map features to disease categories

**Q4. CNN Structure:**
```
INPUT IMAGE → [Conv2D → ReLU → BatchNorm → MaxPool] × N → Flatten → Dense → Softmax → OUTPUT CLASS
```